# Arctic‑tuned ocean color over DBO — **Lewis & Arrigo (2020)** from MODIS Rrs (DAILY)

The standard OBPG products (OCx chl, GIOP adg/bbp) are globally tuned and biased in the high‑CDOM
Arctic. Here we stream **MODIS‑Aqua remote‑sensing reflectance** (`MODISA_L3m_RRS`) over DBO and
compute chl / adg(443) / bbp(443) with the **region‑specific Lewis & Arrigo (2020)** algorithms,
then run the same DBO box time‑series + trend analysis and **validate against the standard products**.

### Algorithms (coefficients from Lewis & Arrigo 2020, JGR e2019JC015706)
- **chl — `AOReg.emp`** (max band‑ratio, inflow/outflow shelf coeffs; DBO = Pacific inflow shelf):
  `Chl = 10^(a + bR + cR²)`, R = log10(max(Rrs443,Rrs488)/Rrs547), **a=0.2391, b=−3.3662, c=1.1085**.
- **adg(443), bbp(443), chl — `AO.GSM`**: GSM semi‑analytical inversion of the Rrs spectrum with
  Arctic params **Sdg=0.018996, Sbbp=1.3309**, Arctic `a*ph` spectrum; L&A eq 10 `Rrs=0.544·(f/Q)·u` (f/Q=0.0949), Pope&Fry aw, Morel bbw. Inversion validated by round‑trip.
  **chl and bbp are the trustworthy GSM outputs; `adg` is UNRELIABLE here** — the large Arctic `a*ph`
  lets phytoplankton absorption mimic CDOM, so the inversion collapses adg toward 0. Use the GIOP `adg`
  product (your adg notebook) for CDOM; treat AO.GSM adg as diagnostic only.

### Notes
- Per‑pixel chl is cheap (vectorized band ratio); per‑pixel GSM (adg/bbp) is run on **annual‑mean Rrs
  composites** (vectorized Gauss‑Newton, ~20 s) for the maps, and per‑box on box‑mean Rrs for the time series.
- 6 Rrs bands per time step → set `PERIOD="MO"` (281 steps, ~6× lighter) for a first run; `"8D"` for full cadence.

## 1. Imports

In [1]:
import time as _time, warnings, gc, pickle
from pathlib import Path
from datetime import date
import numpy as np, pandas as pd, xarray as xr
import earthaccess
from pyresample import AreaDefinition, geometry
from pyresample.kd_tree import resample_nearest
from matplotlib.path import Path as MplPath
import matplotlib.pyplot as plt, matplotlib.dates as mdates
import cartopy.crs as ccrs, cartopy.feature as cfeature
from concurrent.futures import ThreadPoolExecutor, wait, FIRST_COMPLETED
from scipy.optimize import least_squares
from scipy.stats import theilslopes, kendalltau, norm, spearmanr
warnings.filterwarnings("ignore"); print("imports OK")

imports OK


## 2. Configuration — product, Lewis&Arrigo coefficients, EASE‑2 grid, DBO boxes

In [2]:
RRS_SHORT="MODISA_L3m_RRS"; RRS_CONCEPT="C3380709369-OB_CLOUD"
PERIOD="DAY"                                 # DAILY (~8500 steps; matches the original MODIS chl analysis)
BANDS=["Rrs_412","Rrs_443","Rrs_488","Rrs_531","Rrs_547","Rrs_555"]   # chl: 443/488/547 (MODIS) | GSM: 412/443/488/531/555
GIDX=[0,1,2,3,5]                                                       # stack indices feeding the 5-band GSM

# --- Lewis & Arrigo 2020 coefficients (chl = MODIS bands; GSM = SeaWiFS 5-band, nearest MODIS Rrs) ---
AOREG=dict(a=0.2391,b=-3.3662,c=1.1085)      # chl AOReg.emp regional inflow/outflow shelf (MODIS 443>488/547)
AOEMP=dict(a=0.0957,b=-2.7973,c=0.6581)      # chl AO.emp pan-Arctic (as used in dbo-insitu-representativeness)
LAMG=np.array([412.,443.,490.,510.,555.])    # AO.GSM model wavelengths (SeaWiFS); MODIS 488/531/555 fed in
AW =np.array([0.004562,0.006953,0.0150,0.0325,0.0596])                 # Pope&Fry aw at GSM wavelengths
BBW=0.5*0.00288*(LAMG/500.)**-4.32                                     # Morel bbw
APH=np.array([0.28503,0.21099,0.089298,0.066926,0.029377])            # AO.GSM a*ph at EXACT wavelengths (no interp)
SDG=0.018996; SBBP=1.3309; FQ=0.0949; TRANS=0.544; BBP_CORR=1.2        # AO.GSM eq 10; bbp/1.2 = L&A ad-hoc bias correction

START_DATE=date(2002,7,1); END_DATE=date.today()
LAT_MIN,LAT_MAX=45,90; ROI=6_000; N_WORKERS=8; DBO_BBOX=(-180.0,50.0,-124.0,74.0)
dbo_area=AreaDefinition("ease2_north_dbo_4km","DBO","EPSG:6931",
    {"proj":"laea","lat_0":90,"lon_0":0,"x_0":0,"y_0":0,"datum":"WGS84","units":"m"},
    906,845,(-3_624_000,992_000,0,4_372_000))
CHECKPOINT=True; CKPT_PATH=Path.home()/f"modis_LA_dbo_{PERIOD}_v2_ckpt.pkl"; CKPT_EVERY=60  # v2 = AOReg.emp + AO.emp per pixel  # add your path
DBO_BOXES={
 "DBO1":[(63.769462,-172.312422),(61.945988,-172.186927),(61.846516,-175.790978),(63.662718,-176.147111)],
 "DBO2":[(65.107747,-170.865176),(65.097001,-167.859549),(64.481828,-167.907758),(64.493658,-170.840669)],
 "DBO3":[(66.785880,-171.330008),(68.608869,-171.419334),(68.572412,-166.481029),(66.752281,-166.755808)],
 "DBO4":[(71.994000,-164.226000),(71.801000,-159.729000),(70.641000,-160.305000),(70.821000,-164.556000)],
 "DBO5":[(71.807730,-158.537691),(71.635583,-155.931220),(71.110941,-156.307277),(71.278277,-158.847920)],
 "DBO6":[(71.850000,-150.108000),(70.890000,-150.976000),(71.159000,-153.865000),(72.119000,-152.980000)],
 "DBO7":[(70.868000,-141.528000),(69.946000,-142.679000),(70.333000,-145.439000),(71.253000,-144.256000)],
 "DBO8":[(70.834000,-125.155000),(70.049000,-126.997000),(70.664000,-129.354000),(71.442000,-127.431000)]}
data_crs=ccrs.LambertAzimuthalEqualArea(central_latitude=90,central_longitude=0)
disp=ccrs.LambertAzimuthalEqualArea(central_latitude=90,central_longitude=-152)
x0,y0,x1,y1=dbo_area.area_extent
print(f"{RRS_SHORT} | {PERIOD} | {START_DATE}->{END_DATE} | {len(BANDS)} bands | AOReg.emp + AO.emp (per pixel) + AO.GSM")

MODISA_L3m_RRS | DAY | 2002-07-01->2026-07-14 | 6 bands | AOReg.emp + AO.emp (per pixel) + AO.GSM


## 3. Auth + search the Rrs band granules

In [3]:
auth=earthaccess.login(persist=True)
def granule_date(g):
    d=g.data_links()[0].rsplit("/",1)[-1].split(".")[1].split("_")[0]
    return date(int(d[:4]),int(d[4:6]),1 if PERIOD=="MO" else int(d[6:8]))
granules_by_period={}      # {date: {band: granule}}
for bn in BANDS:
    pat=f"*.{PERIOD}.RRS.{bn}.4km.nc"
    g=earthaccess.search_data(concept_id=RRS_CONCEPT,granule_name=pat,
                              temporal=(START_DATE.isoformat(),END_DATE.isoformat()),count=12000)
    for x in g: granules_by_period.setdefault(granule_date(x),{})[bn]=x
    print(f"  {bn}: {len(g)} granules")
periods=sorted(d for d,bd in granules_by_period.items() if len(bd)==len(BANDS))   # need all bands
print(f"{len(periods)} time steps with all {len(BANDS)} bands ({periods[0]} -> {periods[-1]})")

  Rrs_412: 10310 granules


## 4. Lewis & Arrigo algorithm module (validated: round‑trip + vectorized GN)

In [4]:
DG=np.exp(-SDG*(LAMG-443.)); BP=(LAMG/443.)**(-SBBP)   # 5-band GSM at SeaWiFS model wavelengths
def aoreg_chl(r443,r488,r547):                # AOReg.emp — regional inflow/outflow shelf
    R=np.log10(np.maximum(r443,r488)/r547)
    return 10**(AOREG["a"]+AOREG["b"]*R+AOREG["c"]*R*R)
def aoemp_chl(r443,r488,r547):                # AO.emp — pan-Arctic (same band ratio, different coeffs)
    R=np.log10(np.maximum(r443,r488)/r547)
    return 10**(AOEMP["a"]+AOEMP["b"]*R+AOEMP["c"]*R*R)
def _fwd(P):                                  # P:(N,3) log[chl,adg,bbp] -> Rrs (N,5)
    chl,adg,bbp=np.exp(P).T
    a=AW+chl[:,None]*APH+adg[:,None]*DG; bb=BBW+bbp[:,None]*BP
    return TRANS*FQ*(bb/(a+bb))           # L&A 2020 eq 10 (linear in u, NOT Gordon g1/g2)
def gsm_box(rrs):                             # robust scalar inversion (per box spectrum)
    r=least_squares(lambda p:_fwd(p[None,:])[0]-rrs,np.log([1.,0.02,0.003]),method="lm",max_nfev=300)
    v=np.exp(r.x); v[2]/=BBP_CORR; return v   # chl, adg443, bbp443 (bbp/1.2 = L&A correction)
def gsm_grid(RRS,iters=20):                   # vectorized Gauss-Newton over many pixels (N,5)->(N,3)
    ok=np.all(np.isfinite(RRS)&(RRS>0),axis=1); out=np.full((len(RRS),3),np.nan)
    if not ok.any(): return out
    X=RRS[ok]; P=np.tile(np.log([1.,0.02,0.003]),(len(X),1)); eps=1e-4
    for _ in range(iters):
        r=_fwd(P)-X; J=np.empty((len(X),X.shape[1],3))
        for k in range(3):
            dP=P.copy(); dP[:,k]+=eps; J[:,:,k]=(_fwd(dP)-_fwd(P))/eps
        JtJ=np.einsum("nik,nij->nkj",J,J)+1e-6*np.eye(3); Jtr=np.einsum("nik,ni->nk",J,r)
        P=np.clip(P-np.linalg.solve(JtJ,Jtr),np.log(1e-4),np.log(50))
    out[ok]=np.exp(P); out[:,2]/=BBP_CORR; return out   # bbp/1.2 = L&A correction
print("algorithm ready: aoreg_chl, aoemp_chl, gsm_box, gsm_grid")

algorithm ready: aoreg_chl, aoemp_chl, gsm_box, gsm_grid


## 5. Reader — one time step → 6 Rrs band grids on the EASE‑2 DBO grid

In [5]:
def _read_band(g):
    f=earthaccess.open([g],show_progress=False)[0]
    try:
        with xr.open_dataset(f,engine="h5netcdf") as ds:
            bn=[v for v in ds.data_vars if v.startswith("Rrs_")][0]
            lat=np.asarray(ds["lat"].values,float); lon=np.asarray(ds["lon"].values,float); lo0,la0,lo1,la1=DBO_BBOX
            jl=np.where((lat>=la0)&(lat<=la1))[0]; il=np.where((lon>=lo0)&(lon<=lo1))[0]
            sub=ds[bn].isel(lat=slice(int(jl[0]),int(jl[-1])+1),lon=slice(int(il[0]),int(il[-1])+1))
            v=np.asarray(sub.values,float).squeeze(); la=np.asarray(sub["lat"].values,float); lo=np.asarray(sub["lon"].values,float)
    finally:
        try: f.close()
        except Exception: pass
    return la,lo,v
def project_rrs(band_granules):
    """-> (6, ny, nx) stack of Rrs on dbo_area (NaN where missing)."""
    out=np.full((len(BANDS),)+dbo_area.shape,np.nan,np.float32)
    for bi,bn in enumerate(BANDS):
        la,lo,v=_read_band(band_granules[bn]); lon2d,lat2d=np.meshgrid(lo,la); m=np.isfinite(v)
        if not m.any(): continue
        sw=geometry.SwathDefinition(lons=np.where(lon2d>180,lon2d-360,lon2d)[m],lats=lat2d[m])
        out[bi]=resample_nearest(sw,v[m],dbo_area,radius_of_influence=ROI,fill_value=np.nan,reduce_data=False)
    return out

## 6. DBO box masks

In [6]:
lons2d,lats2d=dbo_area.get_lonlats(); gp=np.column_stack([lons2d.ravel(),lats2d.ravel()])
box_masks={n:MplPath([(lo,la) for la,lo in c]+[(c[0][1],c[0][0])]).contains_points(gp).reshape(lons2d.shape)
           for n,c in DBO_BOXES.items()}
for n,m in box_masks.items(): print(f"  {n}: {int(m.sum()):>6,} cells")

  DBO1:  2,412 cells
  DBO2:    605 cells
  DBO3:  2,561 cells
  DBO4:  1,293 cells
  DBO5:    351 cells
  DBO6:    739 cells
  DBO7:    773 cells
  DBO8:    760 cells


## 7. Streaming pass — AOReg.emp chl per pixel + accumulate Rrs (checkpointed)

Per time step: read 6 Rrs grids → AOReg.emp chl per pixel (folded to box series + annual grid + mean),
and accumulate the annual‑mean Rrs grids and per‑box mean Rrs spectra for the GSM steps.

In [7]:
ny,nx=dbo_area.shape; NB=len(BANDS)
def _have(): return "rrs_box" in globals() and bool(rrs_box)
if not _have() and CHECKPOINT and CKPT_PATH.exists():
    _ck=pickle.load(open(CKPT_PATH,"rb"))
    (chl_rows,rrs_box,rrs_ann,chl_msum,chl_mcnt,chl_ann,
     chl_rows_e,chl_msum_e,chl_mcnt_e,chl_ann_e,done) = (_ck[k] for k in
        ["chl_rows","rrs_box","rrs_ann","chl_msum","chl_mcnt","chl_ann",
         "chl_rows_e","chl_msum_e","chl_mcnt_e","chl_ann_e","done"])
    print(f"resumed {len(done)} steps")
elif not _have():
    chl_rows={}; rrs_box={}; rrs_ann={}; chl_ann={}; chl_rows_e={}; chl_ann_e={}; done=set()
    chl_msum  =np.zeros((ny,nx),np.float32); chl_mcnt  =np.zeros((ny,nx),np.float32)   # AOReg.emp per-pixel accum
    chl_msum_e=np.zeros((ny,nx),np.float32); chl_mcnt_e=np.zeros((ny,nx),np.float32)   # AO.emp   per-pixel accum

def _one(item):
    i,d=item
    try: return i,d,project_rrs(granules_by_period[d]),None
    except Exception as e: return i,d,None,f"{type(e).__name__}:{str(e)[:50]}"
def _acc(chl,m,msum,mcnt,annd,d):                          # fold one algorithm's per-pixel daily chl
    msum[m]+=chl[m]; mcnt[m]+=1
    s=annd.setdefault(d.year,[np.zeros((ny,nx),np.float32),np.zeros((ny,nx),np.float32)]); s[0][m]+=chl[m]; s[1][m]+=1
def _fold(d,R):
    r443,r488,r547=R[1],R[2],R[4]
    chl =aoreg_chl(r443,r488,r547); m =np.isfinite(chl )&(chl >0)         # AOReg.emp per pixel
    chle=aoemp_chl(r443,r488,r547); me=np.isfinite(chle)&(chle>0)         # AO.emp   per pixel (same daily Rrs)
    _acc(chl ,m ,chl_msum ,chl_mcnt ,chl_ann ,d)
    _acc(chle,me,chl_msum_e,chl_mcnt_e,chl_ann_e,d)
    a=rrs_ann.setdefault(d.year,[np.zeros((NB,ny,nx),np.float32),np.zeros((NB,ny,nx),np.float32)])
    for bi in range(NB):
        mb=np.isfinite(R[bi]); a[0][bi][mb]+=R[bi][mb]; a[1][bi][mb]+=1
    chl_rows  [d]={n:(float(np.nanmean(chl [mm])) if (mm:=box_masks[n]&m ).any() else np.nan) for n in box_masks}
    chl_rows_e[d]={n:(float(np.nanmean(chle[mm])) if (mm:=box_masks[n]&me).any() else np.nan) for n in box_masks}
    rrs_box[d]={n:[float(np.nanmean(R[bi][box_masks[n]&np.isfinite(R[bi])])) if (box_masks[n]&np.isfinite(R[bi])).any()
                   else np.nan for bi in range(NB)] for n in box_masks}
    done.add(d)
def _save():
    if not CHECKPOINT: return
    tmp=CKPT_PATH.with_suffix(".tmp")
    pickle.dump(dict(chl_rows=chl_rows,rrs_box=rrs_box,rrs_ann=rrs_ann,chl_msum=chl_msum,chl_mcnt=chl_mcnt,chl_ann=chl_ann,
                     chl_rows_e=chl_rows_e,chl_msum_e=chl_msum_e,chl_mcnt_e=chl_mcnt_e,chl_ann_e=chl_ann_e,done=done),
                open(tmp,"wb")); tmp.replace(CKPT_PATH)

todo=[(i,d) for i,d in enumerate(periods) if d not in done]; t0=_time.time(); nok=nf=since=0
print(f"{len(done)} done | streaming {len(todo)} steps x {NB} bands (AOReg.emp + AO.emp per pixel)...")
with ThreadPoolExecutor(max_workers=N_WORKERS) as ex:
    fs={ex.submit(_one,it) for it in todo[:N_WORKERS*2]}; it=iter(todo[N_WORKERS*2:])
    while fs:
        dn,fs=wait(fs,return_when=FIRST_COMPLETED)
        for fut in dn:
            i,d,R,err=fut.result()
            if R is None: nf+=1; (print("  X",d,err) if nf<=8 else None)
            else:
                _fold(d,R); nok+=1; since+=1
                if nok%25==0: print(f"  {nok} ok / {nf} fail ({(_time.time()-t0)/60:.1f} min)")
                if since>=CKPT_EVERY: _save(); since=0
            try: fs.add(ex.submit(_one,next(it)))
            except StopIteration: pass
_save()
mean_chl  =np.where(chl_mcnt  >0,chl_msum  /chl_mcnt  ,np.nan)   # AOReg.emp per-pixel record mean (mean of daily chl)
mean_chl_e=np.where(chl_mcnt_e>0,chl_msum_e/chl_mcnt_e,np.nan)   # AO.emp   per-pixel record mean (mean of daily chl)
print(f"folded {len(done)} steps. chl boxes with data: {[n for n in box_masks if np.isfinite([chl_rows[d][n] for d in chl_rows]).any()]}")

0 done | streaming 8530 steps x 6 bands (AOReg.emp + AO.emp per pixel)...
  25 ok / 0 fail (0.7 min)
  50 ok / 0 fail (1.2 min)
  75 ok / 0 fail (1.7 min)
  100 ok / 0 fail (2.1 min)
  125 ok / 0 fail (2.5 min)
  150 ok / 0 fail (2.7 min)
  175 ok / 0 fail (3.0 min)
  200 ok / 0 fail (3.3 min)
  225 ok / 0 fail (3.6 min)
  250 ok / 0 fail (4.1 min)
  275 ok / 0 fail (4.5 min)
  300 ok / 0 fail (5.0 min)
  325 ok / 0 fail (5.4 min)
  350 ok / 0 fail (5.9 min)
  375 ok / 0 fail (6.4 min)
  400 ok / 0 fail (6.8 min)
  425 ok / 0 fail (7.3 min)
  450 ok / 0 fail (7.8 min)
  475 ok / 0 fail (8.2 min)
  500 ok / 0 fail (8.6 min)
  525 ok / 0 fail (8.8 min)
  550 ok / 0 fail (9.0 min)
  575 ok / 0 fail (9.4 min)
  600 ok / 0 fail (9.8 min)
  625 ok / 0 fail (10.3 min)
  650 ok / 0 fail (10.7 min)
  675 ok / 0 fail (11.2 min)
  700 ok / 0 fail (11.6 min)
  725 ok / 0 fail (12.1 min)
  750 ok / 0 fail (12.5 min)
  775 ok / 0 fail (13.1 min)
  800 ok / 0 fail (13.6 min)
  825 ok / 0 fail (14.0 m